In [19]:

import pandas as pd
import numpy as np

from pathlib import Path
import sys

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import roc_auc_score

from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

import joblib

# Import our separate feature engineering function
current_dir = Path.cwd()

if (current_dir / "features.py").exists():
    sys.path.insert(0, str(current_dir))
else:
    sys.path.insert(0, str(current_dir / "week3"))

from features import add_features

# Load hotel booking dataset
url = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-11/hotels.csv"

df = pd.read_csv(url)

print("Original dataset shape:", df.shape)

# Remove columns unavailable at booking time
df = df.drop(columns=[
    "reservation_status",
    "reservation_status_date",
    "assigned_room_type",
    "booking_changes"
])

print("Dataset after dropping columns:", df.shape)


Original dataset shape: (119390, 32)
Dataset after dropping columns: (119390, 28)


The original dataset contains 119,390 bookings and 32 columns. After removing 4 unavailable or leakage-prone columns, 28 columns remain.

In [20]:

# Separate features and target
X = df.drop(columns=["is_canceled"])
y = df["is_canceled"]

# Required stratified train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True).round(4))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True).round(4))


Training features: (95512, 27)
Testing features: (23878, 27)

Training target distribution:
is_canceled
0    0.6296
1    0.3704
Name: proportion, dtype: float64

Testing target distribution:
is_canceled
0    0.6296
1    0.3704
Name: proportion, dtype: float64


The stratified split produced 95,512 training bookings and 23,878 testing bookings, preserving approximately the same cancellation distribution.

In [21]:

# Apply our feature engineering function
X_train_features = add_features(X_train)

# Check the five new features
new_features = [
    "total_nights",
    "total_guests",
    "adr_per_guest",
    "arrival_weekday",
    "prev_cancel_rate"
]

print("New features created:", new_features)

print("\nTraining data shape after feature engineering:")
print(X_train_features.shape)

print("\nFeature examples:")
print(X_train_features[new_features].head())

print("\nMissing children values before handling:")
print(X_train["children"].isna().sum())

print("\nMissing agent values after handling:")
print(X_train_features["agent"].isna().sum())


New features created: ['total_nights', 'total_guests', 'adr_per_guest', 'arrival_weekday', 'prev_cancel_rate']

Training data shape after feature engineering:
(95512, 32)

Feature examples:
        total_nights  total_guests  adr_per_guest  arrival_weekday  \
105447             2           1.0           98.0                1   
85242              3           2.0           50.0                2   
65604              3           2.0           47.5                3   
17345             14           2.0           27.0                2   
117786             1           1.0           80.0                2   

        prev_cancel_rate  
105447               0.0  
85242                0.0  
65604                0.0  
17345                0.0  
117786               0.0  

Missing children values before handling:
4

Missing agent values after handling:
0


Feature engineering added 5 columns, increasing the training features from 27 to 32, and missing agent values were handled.

In [22]:

# Numeric preprocessing
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical/text preprocessing
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both preprocessing pipelines
preprocessor = ColumnTransformer(
    transformers=[
        ("numbers", numeric_transformer,
         make_column_selector(dtype_include=np.number)),

        ("text", categorical_transformer,
         make_column_selector(dtype_include=object))
    ]
)

print("Numeric preprocessing: Median imputation + StandardScaler")
print("Text preprocessing: Missing-value imputation + OneHotEncoder")
print("Preprocessor created successfully.")


Numeric preprocessing: Median imputation + StandardScaler
Text preprocessing: Missing-value imputation + OneHotEncoder
Preprocessor created successfully.


I created separate preprocessing pipelines for numeric and text columns, using median imputation and scaling for numbers and missing-value imputation and one-hot encoding for text.

In [23]:

def build_pipeline(model, use_features=True):

    # Apply feature engineering when requested
    if use_features:
        feature_step = FunctionTransformer(add_features)
    else:
        feature_step = "passthrough"

    pipeline = Pipeline(steps=[
        ("features", feature_step),
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    return pipeline


print("Reusable machine learning pipeline created.")


Reusable machine learning pipeline created.


In [24]:

# Model 1: Logistic Regression without engineered features
logistic_without_features = build_pipeline(
    LogisticRegression(max_iter=2000),
    use_features=False
)

logistic_without_features.fit(X_train, y_train)

# Test predictions
pred_without_features = logistic_without_features.predict_proba(X_test)[:, 1]

auc_without_features = roc_auc_score(
    y_test,
    pred_without_features
)

# Model 2: Logistic Regression with engineered features
logistic_with_features = build_pipeline(
    LogisticRegression(max_iter=2000),
    use_features=True
)

logistic_with_features.fit(X_train, y_train)

# Test predictions
pred_with_features = logistic_with_features.predict_proba(X_test)[:, 1]

auc_with_features = roc_auc_score(
    y_test,
    pred_with_features
)

# Compare results
logistic_results = pd.DataFrame({
    "Model": [
        "Without Engineered Features",
        "With Engineered Features"
    ],
    "Test ROC AUC": [
        auc_without_features,
        auc_with_features
    ]
})

print(logistic_results.to_string(index=False))

difference = auc_with_features - auc_without_features

print("\nAUC improvement:", round(difference, 5))
print("Improvement in percentage points:", round(difference * 100, 2))


                      Model  Test ROC AUC
Without Engineered Features      0.887396
   With Engineered Features      0.907450

AUC improvement: 0.02005
Improvement in percentage points: 2.01


Logistic Regression achieved a test ROC AUC of 0.887396 without engineered features and 0.907450 with them. The five engineered features improved the ROC AUC by 0.02005 (approximately 2.01 percentage points).

In [25]:
xgb_pipeline = build_pipeline(
    XGBClassifier(
        n_estimators=300,
        random_state=42,
        eval_metric="logloss",
        n_jobs=1
    ),
    use_features=True
)

param_grid = {
    "model__max_depth": [4, 6, 8],
    "model__learning_rate": [0.05, 0.1, 0.2]
}

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

grid_search = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=2,
    verbose=1,
    return_train_score=True
)

print("XGBoost pipeline configured.")
print("Hyperparameter combinations:", 9)
print("Cross-validation folds:", 3)
print("Total model fits:", 27)

XGBoost pipeline configured.
Hyperparameter combinations: 9
Cross-validation folds: 3
Total model fits: 27


I configured XGBoost with 300 estimators and nine hyperparameter combinations using three-fold stratified cross-validation.

In [26]:
grid_search.fit(X_train, y_train)

print("\nBest parameters:")
print(grid_search.best_params_)

print("\nBest cross-validation ROC AUC:")
print(round(grid_search.best_score_, 5))

Fitting 3 folds for each of 9 candidates, totalling 27 fits

Best parameters:
{'model__learning_rate': 0.2, 'model__max_depth': 8}

Best cross-validation ROC AUC:
0.9526


GridSearchCV evaluated 9 hyperparameter combinations across 3 folds and identified the best settings with a cross-validation ROC AUC of 0.9526


In [27]:

# Extract all GridSearchCV results
cv_results = pd.DataFrame(grid_search.cv_results_)

# Create a clean table with the required columns
results_table = cv_results[
    [
        "param_model__max_depth",
        "param_model__learning_rate",
        "mean_test_score",
        "std_test_score"
    ]
].copy()

results_table.columns = [
    "Max Depth",
    "Learning Rate",
    "Mean CV ROC AUC",
    "Std CV ROC AUC"
]

# Sort by mean CV ROC AUC
results_table = results_table.sort_values(
    by="Mean CV ROC AUC",
    ascending=False
).reset_index(drop=True)

print(results_table.to_string(index=False))


 Max Depth  Learning Rate  Mean CV ROC AUC  Std CV ROC AUC
         8           0.20         0.952603        0.000747
         6           0.20         0.949712        0.000287
         8           0.10         0.949594        0.000680
         8           0.05         0.946508        0.000346
         6           0.10         0.945669        0.000664
         4           0.20         0.944683        0.000421
         6           0.05         0.942316        0.000331
         4           0.10         0.940648        0.000477
         4           0.05         0.936032        0.000545


The results table compares all nine hyperparameter combinations using mean and standard deviation of three-fold cross-validation ROC AUC. The highest mean score identifies the best combination.

In [28]:

# Get the best model selected by GridSearchCV
best_xgb_model = grid_search.best_estimator_

# Predict cancellation probabilities on untouched test data
best_test_predictions = best_xgb_model.predict_proba(X_test)[:, 1]

# Calculate test ROC AUC
best_test_auc = roc_auc_score(
    y_test,
    best_test_predictions
)

# Calculate training ROC AUC for comparison
best_train_predictions = best_xgb_model.predict_proba(X_train)[:, 1]

best_train_auc = roc_auc_score(
    y_train,
    best_train_predictions
)

print("Best parameters:", grid_search.best_params_)
print("Best CV ROC AUC:", round(grid_search.best_score_, 5))
print("Test ROC AUC:", round(best_test_auc, 5))
print("Training ROC AUC:", round(best_train_auc, 5))


Best parameters: {'model__learning_rate': 0.2, 'model__max_depth': 8}
Best CV ROC AUC: 0.9526
Test ROC AUC: 0.95478
Training ROC AUC: 0.97658


The best XGBoost configuration achieved a CV ROC AUC of 0.9526 and a test ROC AUC of 0.95478, while its training ROC AUC was 0.97658.

In [29]:

best_params = grid_search.best_params_

best_depth = best_params["model__max_depth"]
best_lr = best_params["model__learning_rate"]

at_boundary = (
    best_depth in [4, 8]
    or best_lr in [0.05, 0.2]
)

print("Best max_depth:", best_depth)
print("Best learning_rate:", best_lr)

if at_boundary:
    print("The best setting is at the edge of the grid.")
    print("An additional experiment is required.")
else:
    print("The best setting is inside the grid.")
    print("No edge extension is required.")


Best max_depth: 8
Best learning_rate: 0.2
The best setting is at the edge of the grid.
An additional experiment is required.


I checked whether the best hyperparameters were at the grid boundary and tested one step beyond the selected boundary. I compared the additional experiment's CV, test and training AUC before selecting the final model.

In [30]:

if at_boundary:

    extra_param_grid = []

    # Extend max_depth by one step beyond the selected boundary
    if best_depth == 8:
        extra_param_grid.append({
            "model__max_depth": [10],
            "model__learning_rate": [best_lr]
        })

    elif best_depth == 4:
        extra_param_grid.append({
            "model__max_depth": [3],
            "model__learning_rate": [best_lr]
        })

    # Extend learning_rate by one step beyond the selected boundary
    if best_lr == 0.2:
        extra_param_grid.append({
            "model__max_depth": [best_depth],
            "model__learning_rate": [0.3]
        })

    elif best_lr == 0.05:
        extra_param_grid.append({
            "model__max_depth": [best_depth],
            "model__learning_rate": [0.025]
        })

    extended_search = GridSearchCV(
        estimator=xgb_pipeline,
        param_grid=extra_param_grid,
        scoring="roc_auc",
        cv=cv,
        n_jobs=2,
        verbose=1,
        return_train_score=True
    )

    extended_search.fit(X_train, y_train)

    extended_model = extended_search.best_estimator_

    extended_test_auc = roc_auc_score(
        y_test,
        extended_model.predict_proba(X_test)[:, 1]
    )

    extended_train_auc = roc_auc_score(
        y_train,
        extended_model.predict_proba(X_train)[:, 1]
    )

    print("Extended settings:", extended_search.best_params_)
    print("Extended CV AUC:", round(extended_search.best_score_, 5))
    print("Extended test AUC:", round(extended_test_auc, 5))
    print("Extended training AUC:", round(extended_train_auc, 5))

    # Select the model with the higher CV AUC
    if extended_search.best_score_ > grid_search.best_score_:
        final_model = extended_model
        print("\nExtended model selected.")
    else:
        final_model = best_xgb_model
        print("\nOriginal grid model retained.")

else:
    final_model = best_xgb_model
    print("No boundary extension required.")
    print("Original best model retained.")


Fitting 3 folds for each of 2 candidates, totalling 6 fits
Extended settings: {'model__learning_rate': 0.2, 'model__max_depth': 10}
Extended CV AUC: 0.95481
Extended test AUC: 0.9575
Extended training AUC: 0.98995

Extended model selected.


I checked whether the best hyperparameters were at the grid boundary and tested one step beyond the selected boundary. I compared the additional experiment's CV, test and training AUC before selecting the final model.

In [31]:

# Locate the week3 folder
output_dir = Path.cwd()

if output_dir.name != "week3":
    output_dir = output_dir / "week3"

output_dir.mkdir(exist_ok=True)

# Save the final trained pipeline
model_path = output_dir / "booking_model.joblib"

joblib.dump(final_model, model_path)

print("Model saved successfully!")
print("File location:", model_path)
print("File exists:", model_path.exists())
print("File size (KB):", round(model_path.stat().st_size / 1024, 2))


Model saved successfully!
File location: f:\web3geeks-ai-bootcamp\week3\booking_model.joblib
File exists: True
File size (KB): 1986.92


I saved the selected, fully trained XGBoost pipeline as booking_model.joblib using joblib.

In [32]:

# Load the saved model
loaded_model = joblib.load(model_path)

# Select one booking from the test dataset
one_booking = X_test.iloc[[0]]

# Predict cancellation probability
cancellation_probability = loaded_model.predict_proba(one_booking)[0, 1]

# Predict class using the default 0.5 threshold
predicted_class = loaded_model.predict(one_booking)[0]

# Actual outcome
actual_class = y_test.iloc[0]

print("Model loaded successfully.")
print("Cancellation probability:", round(cancellation_probability, 4))
print("Predicted class:", predicted_class)
print("Actual class:", actual_class)

print(
    "Prediction:",
    "Cancelled" if predicted_class == 1 else "Not cancelled"
)


Model loaded successfully.
Cancellation probability: 0.1711
Predicted class: 0
Actual class: 0
Prediction: Not cancelled


I successfully reloaded the saved model and generated a cancellation probability for one unseen test booking, confirming that the saved pipeline works.

Our XGBoost model achieved a test ROC AUC of 0.95478, showing its ability to distinguish bookings likely to be cancelled from those likely to be completed.

A booking with a cancellation probability of 0.8 has a high estimated cancellation risk and deserves additional attention.

The hotel manager should consider a confirmation message or follow-up with the customer, rather than automatically cancelling the booking.
